In [0]:
from pyspark.sql import functions as F

In [0]:
silver_table = "hospital_poc.silver.patient_transcripts"

df_silver = spark.table(silver_table)

print("Silver records:", df_silver.count())

display(
    df_silver
    .select("patient_id", "specialty", "transcript")
    .limit(5)
)

Silver records: 1000


patient_id,specialty,transcript
1,oncology,"Patient ID: 1. Specialty: oncology. Patient name: Baby Saanvi. Age/Sex: 68/F. The patient presented with Unexplained weight loss and fatigue for 8 months.. The history of present illness was Patient noticed a gradual loss of appetite.. Relevant past history included No history of major illness.. Physical examination showed 2x2 cm hard lump in the upper outer quadrant of the left breast. - PICCKLE: Pallor: +, Icterus: -, Cyanosis: -, Clubbing: -, Koilonychia: -, Lymphadenopathy: -, Edema: -. Investigations revealed CBC shows anemia. PET scan shows metastasis.. The diagnosis was Metastatic Adenocarcinoma of unknown primary (ICD: C80.0).. During the hospital stay, Underwent mastectomy followed by chemotherapy.. At discharge, the patient's condition was Stable, afebrile.. The patient was advised - Regular follow-up in the oncology OPD. - Dawa time pe lena.."
2,cardiology,"Patient ID: 2. Specialty: cardiology. Patient name: Debashish Roy. Age/Sex: 85/F. The patient presented with Palpitations and dizziness for 3 days.. The history of present illness was Substernal pain on exertion, with diaphoresis. Patient ko fever tha.. Relevant past history included Coronary artery disease, post-stent.. Physical examination showed BP: 160/102 mmHg, Heart: S1/S2 normal, no murmur. - PICCKLE: Pallor: +, Icterus: -, Cyanosis: -, Clubbing: -, Koilonychia: -, Lymphadenopathy: -, Edema: -. Investigations revealed ECG: ST elevation; Trop-I: Positive; Echo: EF 45%; Angiogram: 52% blockage in LAD.. The diagnosis was Atrial Fibrillation (ICD: I48.91).. During the hospital stay, Antiarrhythmics given, rhythm stabilized.. At discharge, the patient's condition was Stable, afebrile.. The patient was advised - Meds: Tab. Aspirin 75mg OD, Tab. Atorvastatin 40mg HS. - pura course complete karna.."
3,general_medicine,"Patient ID: 3. Specialty: general_medicine. Patient name: Jayesh Patel. Age/Sex: 85/F. The patient presented with Abdominal pain for 22 days.. The history of present illness was Productive cough, no hemoptysis.. Relevant past history included Diabetes on insulin.. Physical examination showed Temp: 102°F; Chest: Crepitations. - PICCKLE: Pallor: +, Icterus: -, Cyanosis: -, Clubbing: -, Koilonychia: -, Lymphadenopathy: -, Edema: -. Investigations revealed Endoscopy: Gastritis.. The diagnosis was Community-Acquired Pneumonia (ICD: J18.9).. During the hospital stay, IV antibiotics. Fever subsided.. At discharge, the patient's condition was Stable, afebrile.. The patient was advised - Diet: Bland; Follow-up: 1 week.."
4,pediatrics,"Patient ID: 4. Specialty: pediatrics. Patient name: Imran Khan. Age/Sex: 35/M. The patient presented with Loose motions aur vomiting for 17 days.. The history of present illness was High fever, measles-like rash.. Relevant past history included Recent URI.. Physical examination showed Dehydrated, abdomen soft. - PICCKLE: Pallor: +, Icterus: -, Cyanosis: -, Clubbing: -, Koilonychia: -, Lymphadenopathy: -, Edema: -. Investigations revealed CRP: Positive.. The diagnosis was Viral Exanthem (ICD: B09).. During the hospital stay, ORS and IV fluids. Zinc given.. At discharge, the patient's condition was Stable, afebrile.. The patient was advised - Diet: Breastfeed; Follow-up: 5 days. - report leke aana.."
5,pediatrics,"Patient ID: 5. Specialty: pediatrics. Patient name: Sanjay Yadav. Age/Sex: Newborn/M. The patient presented with Fever and rash for 27 days.. The history of present illness was Acute, with mild dehydration.. Relevant past history included Recent URI.. Physical examination showed Temp: 102°F, throat congested. - PICCKLE: Pallor: +, Icterus: -, Cyanosis: -, Clubbing: -, Koilonychia: -, Lymphadenopathy: -, Edema: -. Investigations revealed CRP: Positive.. The diagnosis was Viral Exanthem (ICD: B09).. During the hospital stay, Symptomatic treatment, resolved.. At discharge, the patient's condition was Stable, afebrile.. The patient was advised - Diet: Breastfeed; Follow-up: 5 days

In [0]:
clinical_prompt = """
You are a clinical summarization assistant.

Summarize the provided clinical transcript in less than 300 words.

Include only information that is explicitly present in the transcript:

1. Key medical conditions and diagnoses
2. Active medications, if documented
3. Significant laboratory/investigation findings
4. Important vitals or examination findings, if documented
5. Hospital course and treatment
6. Condition at discharge
7. Follow-up advice or next steps

IMPORTANT RULES:
- Use ONLY information provided in the transcript.
- Do NOT infer, assume, or invent any medical information.
- Do NOT create medications, laboratory values, vitals, diagnoses, or treatments that are not documented.
- If an item is not documented, write "Not documented."
- Keep the summary concise and clinically factual.
- Do not add information from outside knowledge.

Clinical transcript:
"""

In [0]:
%sql

SELECT
    patient_id,
    specialty,
    ai_query(
        'system.ai.gpt-oss-20b',

        CONCAT(
            'You are a clinical summarization assistant.

Summarize the provided clinical transcript in less than 300 words.

Use the following sections:

1. Key medical conditions and diagnoses
2. Active medications, if documented
3. Significant laboratory/investigation findings
4. Important vitals or examination findings, if documented
5. Hospital course and treatment
6. Condition at discharge
7. Follow-up advice or next steps

STRICT RULES:
- Use ONLY information explicitly present in the transcript.
- Do NOT infer or assume information.
- Do NOT invent medications, laboratory values, vitals, diagnoses, treatments, or follow-up plans.
- If information is not documented, write "Not documented."
- Keep the summary concise and clinically factual.
- Maximum 300 words.

Clinical transcript:

',
            transcript
        ),

        modelParameters => named_struct(
            'max_tokens', 600,
            'temperature', 0.0
        ),

        failOnError => false
    ) AS generated_summary

FROM hospital_poc.silver.patient_transcripts

ORDER BY CAST(patient_id AS INT)

LIMIT 5;

patient_id,specialty,generated_summary
1,oncology,"List(**1. Key medical conditions and diagnoses** , null)"
2,cardiology,"List(, null)"
3,general_medicine,"List(**1. Key medical conditions and diagnoses** - Abdominal pain (22 days) - Productive cough, no hemoptysis - Diabetes mellitus (on insulin) - Gastritis (endoscopy) - Community‑Acquired Pneumonia (ICD‑10 J18.9) **2. Active medications, if documented** - Insulin **3. Significant laboratory/investigation findings** - Endoscopy: gastritis **4. Important vitals or examination findings, if documented** - Temperature: 102 °F (fever) - Chest: crepitations - Physical exam: pallor (+); icterus, cyanosis, clubbing, koilonychia, lymphadenopathy, edema – all negative **5. Hospital course and treatment** - Initiated IV antibiotics; fever subsided during stay. **6. Condition at discharge** - Stable, afebrile. **7. Follow‑up advice or next steps** - Bland diet advised. - Follow‑up appointment scheduled in 1 week., null)"
4,pediatrics,"List(**1. Key medical conditions and diagnoses** - Viral exanthem (ICD‑10: B09) **2. Active medications, if documented** - Oral rehydration solution (ORS) - Intravenous fluids - Zinc supplementation **3. Significant laboratory/investigation findings** - C‑reactive protein (CRP): positive **4. Important vitals or examination findings, if documented** - Dehydrated - Abdomen: soft - Pallor: present - Icterus, cyanosis, clubbing, koilonychia, lymphadenopathy, edema: absent **5. Hospital course and treatment** - Admitted for 17 days of loose motions and vomiting. - Managed with OR, null)"
5,pediatrics,"List(**1. Key medical conditions and diagnoses** - Viral Exanthem (ICD‑10: B09) - Mild dehydration - Recent upper respiratory infection **2. Active medications, if documented** Not documented. **3. Significant laboratory/investigation findings** - C‑reactive protein (CRP): Positive **4. Important vitals or examination findings, if documented** - Temperature: 102 °F - Throat: Congested - Pallor: Positive - Icterus, cyanosis, clubbing, koilonychia, lymphadenopathy, edema: Negative **5. Hospital course and treatment** - Symptomatic treatment administered. - Condition resolved during stay. **6. Condition at discharge** - Stable, afebrile. **7. Follow‑up advice or next steps** - Diet: Breastfeed. - Follow‑up appointment: 5 days., null)"


In [0]:
%sql

SELECT
    patient_id,
    specialty,
    ai_query(
        'system.ai.gpt-oss-20b',

        CONCAT(
            'Summarize this clinical transcript.

Return ONLY a concise clinical summary.
Maximum 300 words.

Use ONLY information explicitly present in the transcript.
Do NOT infer, assume, or invent any information.
Do NOT add medications, laboratory values, vitals, diagnoses, treatments, or follow-up plans that are not documented.
If something is not documented, write "Not documented".

Include:
- Key diagnoses
- Active medications
- Significant investigations/labs
- Important vitals/examination findings
- Hospital course/treatment
- Condition at discharge
- Follow-up/next steps

Clinical transcript:

',
            transcript
        ),

        modelParameters => named_struct(
            'max_tokens', 800,
            'temperature', 0.0
        ),

        responseFormat => '{
          "type": "json_schema",
          "json_schema": {
            "name": "clinical_summary",
            "schema": {
              "type": "object",
              "properties": {
                "diagnoses": {"type": "string"},
                "medications": {"type": "string"},
                "investigations": {"type": "string"},
                "vitals_and_exam": {"type": "string"},
                "hospital_course": {"type": "string"},
                "discharge_condition": {"type": "string"},
                "next_steps": {"type": "string"},
                "summary": {"type": "string"}
              },
              "required": [
                "diagnoses",
                "medications",
                "investigations",
                "vitals_and_exam",
                "hospital_course",
                "discharge_condition",
                "next_steps",
                "summary"
              ]
            },
            "strict": true
          }
        }',

        failOnError => false
    ) AS generated_summary

FROM hospital_poc.silver.patient_transcripts

WHERE CAST(patient_id AS INT) = 1;

patient_id,specialty,generated_summary
1,oncology,"List({""medications"": ""Not documented"", ""diagnoses"": ""Metastatic Adenocarcinoma of unknown primary (ICD: C80.0)"", ""hospital_course"": ""Underwent mastectomy followed by chemotherapy"", ""investigations"": ""CBC: anemia; PET scan: metastasis"", ""vitals_and_exam"": ""2x2 cm hard lump in upper outer quadrant of left breast; Pallor +; Icterus -; Cyanosis -; Clubbing -; Koilonychia -; Lymphadenopathy -; Edema -"", ""next_steps"": ""Regular follow-up in the oncology OPD"", ""discharge_condition"": ""Stable, afebrile"", ""summary"": ""68‑year‑old female with 8‑month history of unexplained weight loss and fatigue, found to have a 2x2 cm hard lump in the left breast. CBC revealed anemia; PET scan showed metastatic disease. Diagnosed with metastatic adenocarcinoma of unknown primary (ICD C80.0). Underwent mastectomy and chemotherapy during hospitalization. At discharge, patient was stable and afebrile. Follow‑up: regular oncology outpatient visits.""}, null)"


In [0]:
%sql
---genrate summaries for all 1000
CREATE OR REPLACE TABLE hospital_poc.silver.llm_summary_results AS

SELECT
    patient_id,
    specialty,

    ai_query(
        'system.ai.gpt-oss-20b',

        CONCAT(
            'Summarize this clinical transcript.

Return ONLY the requested structured clinical summary.

Use ONLY information explicitly present in the transcript.
Do NOT infer, assume, or invent any information.
Do NOT add medications, laboratory values, vitals, diagnoses,
treatments, or follow-up plans that are not documented.
If something is not documented, write "Not documented".

The final summary must be less than 300 words.

Clinical transcript:

',
            transcript
        ),

        modelParameters => named_struct(
            'max_tokens', 800,
            'temperature', 0.0
        ),

        responseFormat => '{
          "type": "json_schema",
          "json_schema": {
            "name": "clinical_summary",
            "schema": {
              "type": "object",
              "properties": {
                "diagnoses": {"type": "string"},
                "medications": {"type": "string"},
                "investigations": {"type": "string"},
                "vitals_and_exam": {"type": "string"},
                "hospital_course": {"type": "string"},
                "discharge_condition": {"type": "string"},
                "next_steps": {"type": "string"},
                "summary": {"type": "string"}
              },
              "required": [
                "diagnoses",
                "medications",
                "investigations",
                "vitals_and_exam",
                "hospital_course",
                "discharge_condition",
                "next_steps",
                "summary"
              ],
              "additionalProperties": false
            },
            "strict": true
          }
        }',

        failOnError => false

    ) AS generated_summary,

    current_timestamp() AS generated_at

FROM hospital_poc.silver.patient_transcripts;

num_affected_rows,num_inserted_rows


In [0]:
%sql

SELECT COUNT(*) AS total_records
FROM hospital_poc.silver.llm_summary_results;

total_records
1000


In [0]:
%sql

SELECT
    COUNT(*) AS total,
    SUM(
        CASE
            WHEN generated_summary.errorMessage IS NOT NULL
            THEN 1
            ELSE 0
        END
    ) AS errors
FROM hospital_poc.silver.llm_summary_results;

total,errors
1000,0


In [0]:
%sql

SELECT
    patient_id,
    specialty,
    generated_summary
FROM hospital_poc.silver.llm_summary_results
ORDER BY CAST(patient_id AS INT)
LIMIT 10;

patient_id,specialty,generated_summary
1,oncology,"List({""medications"": ""Not documented"", ""diagnoses"": ""Metastatic Adenocarcinoma of unknown primary (ICD: C80.0)"", ""hospital_course"": ""Underwent mastectomy followed by chemotherapy."", ""investigations"": ""CBC shows anemia. PET scan shows metastasis."", ""vitals_and_exam"": ""Pallor: +, Icterus: -, Cyanosis: -, Clubbing: -, Koilonychia: -, Lymphadenopathy: -, Edema: -."", ""next_steps"": ""Regular follow-up in the oncology OPD."", ""discharge_condition"": ""Stable, afebrile."", ""summary"": ""68‑year‑old female with 8‑month history of unexplained weight loss and fatigue, gradual loss of appetite, and a 2x2 cm hard lump in the left breast. CBC revealed anemia; PET scan showed metastasis. Diagnosis: metastatic adenocarcinoma of unknown primary. Hospital course included mastectomy and chemotherapy. Discharge status: stable, afebrile. Plan: regular oncology follow‑up.""}, null)"
2,cardiology,"List(, null)"
3,general_medicine,"List({""medications"": ""IV antibiotics"", ""diagnoses"": ""Community-Acquired Pneumonia (ICD: J18.9); Gastritis"", ""hospital_course"": ""IV antibiotics administered; fever subsided"", ""investigations"": ""Endoscopy revealed gastritis"", ""vitals_and_exam"": ""Temperature 102°F; chest crepitations; pallor present"", ""next_steps"": ""Diet: bland; follow-up in 1 week"", ""discharge_condition"": ""Stable, afebrile"", ""summary"": ""An 85‑year‑old female presented with 22‑day abdominal pain and productive cough. Physical exam showed fever (102°F) and chest crepitations with pallor. Endoscopy identified gastritis. She was diagnosed with community‑acquired pneumonia (ICD: J18.9). During hospitalization she received IV antibiotics, after which fever resolved. At discharge she was stable and afebrile. Discharge instructions included a bland diet and a follow‑up appointment in one week.""}, null)"
4,pediatrics,"List({""medications"": ""ORS, IV fluids, Zinc"", ""diagnoses"": ""Viral Exanthem (ICD: B09)"", ""hospital_course"": ""Administered ORS, IV fluids, and Zinc during hospital stay."", ""investigations"": ""CRP: Positive"", ""vitals_and_exam"": ""Dehydrated, abdomen soft; Pallor +, Icterus -, Cyanosis -, Clubbing -, Koilonychia -, Lymphadenopathy -, Edema -"", ""next_steps"": ""Diet: Breastfeed; Follow-up: 5 days"", ""discharge_condition"": ""Stable, afebrile"", ""summary"": ""Patient presented with 17 days of loose motions and vomiting, high fever, and measles-like rash. Physical exam showed dehydration, soft abdomen, pallor, and no icterus, cyanosis, clubbing, koilonychia, lymphadenopathy, or edema. CRP was positive. Diagnosed with viral exanthem (ICD B09). Treated with ORS, IV fluids, and zinc. Discharged stable and afebrile, advised breastfeeding and follow-up in 5 days.""}, null)"
5,pediatrics,"List({""medications"": ""Not documented"", ""diagnoses"": ""Viral Exanthem (ICD: B09)"", ""hospital_course"": ""During the hospital stay, symptomatic treatment was provided and the patient resolved."", ""investigations"": ""CRP: Positive"", ""vitals_and_exam"": ""Temperature 102°F, congested throat; pallor present, icterus, cyanosis, clubbing, koilonychia, lymphadenopathy, and edema absent."", ""next_steps"": ""Diet: Breastfeed; follow-up in 5 days."", ""discharge_condition"": ""Stable, afebrile"", ""summary"": ""Newborn male presented with 27 days of fever and rash, mild dehydration, and recent URI. Physical exam showed fever, throat congestion, pallor, and no other abnormalities. CRP was positive. Diagnosis: viral exanthem (ICD B09). Symptomatic treatment was administered during hospitalization, leading to resolution. At discharge, the infant was stable and afebrile. The plan includes continued breastfeeding and a follow-up visit in 5 days.""}, null)"
6,oncology,"List({""medications"": ""palliative chemotherapy"", ""diagnoses"": ""Metastatic Adenocarcinoma of unknown primary (ICD: C80.0)"", ""hospital_course"": ""Patient started on palliative chemotherapy

In [0]:
%sql

DESCRIBE hospital_poc.silver.llm_summary_results;

col_name,data_type,comment
patient_id,bigint,null
specialty,string,null
generated_summary,struct,null
generated_at,timestamp,null
